In [5]:
import requests
import mysql.connector
import time
from datetime import datetime, timedelta

In [6]:
API_KEY = "hBnrFPk07qCcnQy1unyYO2VDgD1nAKH9gi8liBHu"
DB_CONFIG = dict(host="localhost", user="root", password="garvit@123", database="aegis_neo")

In [ ]:
def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

def fetch_neo_feed(start_date, end_date):
    url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date={start_date}&end_date={end_date}&api_key={API_KEY}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()["near_earth_objects"]

def insert_feed_data(data):
    conn = get_conn()
    cur = conn.cursor()
    for date, objects in data.items():
        for obj in objects:
            approach = obj["close_approach_data"][0]
            cur.execute("""
                INSERT IGNORE INTO raw_neo_feed
                (neo_id, name, absolute_magnitude_h, est_diameter_min_km, est_diameter_max_km,
                 is_hazardous, close_approach_date, relative_velocity_kmh, miss_distance_km, orbiting_body)
                VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
            """, (
                obj["id"], obj["name"], obj["absolute_magnitude_h"],
                obj["estimated_diameter"]["kilometers"]["estimated_diameter_min"],
                obj["estimated_diameter"]["kilometers"]["estimated_diameter_max"],
                obj["is_potentially_hazardous_asteroid"], date,
                float(approach["relative_velocity"]["kilometers_per_hour"]),
                float(approach["miss_distance"]["kilometers"]),
                approach["orbiting_body"]
            ))
    conn.commit()
    cur.close(); conn.close()

def fetch_sentry_risk():
    url = "https://ssd-api.jpl.nasa.gov/sentry.api"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json().get("data", [])

def insert_sentry_data(records):
    conn = get_conn()
    cur = conn.cursor()
    for rec in records:
        cur.execute("""
            INSERT INTO raw_sentry_risk
            (neo_id, designation, impact_probability, palermo_scale_max, torino_scale,
             n_impact_events, energy_mt, last_obs_date)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
            ON DUPLICATE KEY UPDATE
              impact_probability=VALUES(impact_probability),
              palermo_scale_max=VALUES(palermo_scale_max)
        """, (
            rec.get("id", rec.get("des")), rec.get("des"), float(rec.get("ip", 0) or 0),
            float(rec.get("ps_max", 0) or 0), int(float(rec.get("ts_max", 0) or 0)),
            int(rec.get("n_imp", 0) or 0), float(rec.get("energy", 0) or 0),
            str(rec.get("last_obs", "")).split(".")[0]
        ))
    conn.commit()
    cur.close(); conn.close()

def fetch_orbital_elements(limit=2000):
    url = "https://ssd-api.jpl.nasa.gov/sbdb_query.api"
    params = {
        "fields": "spkid,full_name,epoch,e,a,i,per,data_arc,n_obs_used",
        "sb-cdata": '{"AND":["neo|EQ|Y"]}',
        "limit": limit
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def insert_orbital_data(payload):
    conn = get_conn()
    cur = conn.cursor()
    for row in payload.get("data", []):
        cur.execute("""
            INSERT INTO raw_orbital_elements
            (neo_id, full_name, epoch, eccentricity, semi_major_axis_au,
             inclination_deg, orbital_period_days, data_arc_days, n_observations)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
            ON DUPLICATE KEY UPDATE epoch=VALUES(epoch)
        """, tuple(row))
    conn.commit()
    cur.close(); conn.close()

In [8]:
if __name__ == "__main__":
    # 1. Pull last 7 days of close approaches (loop this monthly to build history)
    today = datetime.today()
    for i in range(0, 7, 7):
        start = (today - timedelta(days=i+7)).strftime("%Y-%m-%d")
        end = (today - timedelta(days=i)).strftime("%Y-%m-%d")
        insert_feed_data(fetch_neo_feed(start, end))
        time.sleep(1)

    # 2. Pull full risk list
    insert_sentry_data(fetch_sentry_risk())

    # 3. Pull orbital elements for feature engineering later
    insert_orbital_data(fetch_orbital_elements())

    print("Day 1 ingestion complete.")

ProgrammingError: Not enough parameters for the SQL statement